In [17]:
import numpy as np

def f_p(x):
    return np.exp(-1.04855 * x) * (1 + 1.03757 * x + 0.508630 * x**2 + 0.0893988 * x**3)

def b_p(x):
    return np.exp(-1.03149 * x) * (1 + 1.03317 * x + 0.398264 * x**2 + 0.0648056 * x**3)

def f_s(x):
    return np.exp(-1.04190 * x) * (1 + 1.03400 * x + 0.456426 * x**2 + 0.0595248 * x**3)

def b_s(x):
    return np.exp(-1.03365 * x) * (1 + 1.03397 * x + 0.342548 * x**2 + 0.0506182 * x**3)

def S_fit(x):
    return 1 + 7/4 * np.exp(-1.0419 * x) * (1 + 1.034 * x + 0.456426 * x**2 + 0.0595249 * x**3)

def g_starS(T):
    """
    Computes g_{*S}(T) based on the temperature regions defined in the paper.
    
    Parameters:
    T : float or array-like
        Temperature in GeV.
    
    Returns:
    g_starS : float or array-like
        The value of g_{*S} at the given temperature.
    """
    a_i = np.array([1, 1.11724, 3.12672e-1, -4.68049e-2, -2.65004e-2, -1.19706e-3, 
                     1.82812e-4, 1.36436e-4, 8.55015e-5, 1.22840e-5, 3.82259e-7, -6.87035e-9])
    b_i = np.array([1.43382e-2, 1.37559e-2, 2.92108e-3, -5.38533e-4, -1.62496e-4, -2.87906e-5,
                    -3.84278e-6, 2.78776e-6, 7.40342e-7, 1.17210e-7, 3.72499e-9, -6.74107e-11])
    c_i = np.array([1, 6.07869, -1.54485, -2.24034e-1, -2.82147e-2, 2.90620e-2, 
                     6.86778e-3, -1.00005e-3, 1.06301e-2, 1.69528e-6, -9.33311e-8])
    d_i = np.array([7.07388e1, 9.18011e1, 3.31892e1, -1.39779e0, -1.52558e-1, -1.97857e-2, 
                     -1.60146e-6, 8.22615e-5, 2.02651e-5, -1.82134e-5, 7.83943e-8, 7.13518e-5])
    
    T = np.atleast_1d(T)  # Ensure T is an array
    g_starS_values = np.zeros_like(T, dtype=float)
    
    mask_low = T < 0.120
    mask_high = (T >= 0.120) & (T <= 1e16)
    
    if np.any(mask_low):
        m_e, m_mu, m_pi0, m_piPlus = 0.511e-3, 0.1056, 0.135, 0.140  # Masses in GeV
        m_1, m_2, m_3, m_4 = 0.5, 0.77, 1.2, 2  # Masses in GeV
        g_starS_values[mask_low] = (2.008 + 
                                    1.923 * S_fit(m_e / T[mask_low]) + 
                                    3.442 * f_s(m_e / T[mask_low]) + 
                                    3.468 * f_s(m_mu / T[mask_low]) +
                                    1.034 * b_s(m_pi0/ T[mask_low]) +
                                    2.068 * b_s(m_piPlus/ T[mask_low]) +
                                    4.16 * b_s(m_1/ T[mask_low]) +
                                    30.55 * b_s(m_2/ T[mask_low]) +
                                    90 * b_s(m_3/ T[mask_low]) + 
                                    6209 * b_s(m_4/ T[mask_low])
                                   )
    
    if np.any(mask_high):
        t = np.log(T[mask_high])  # Convert to GeV scale
        sum_ai = np.sum(a_i[:, None] * t**np.arange(len(a_i))[:, None], axis=0)
        sum_bi = np.sum(b_i[:, None] * t**np.arange(len(b_i))[:, None], axis=0)
        sum_ci = np.sum(c_i[:, None] * t**np.arange(len(c_i))[:, None], axis=0)
        sum_di = np.sum(d_i[:, None] * t**np.arange(len(d_i))[:, None], axis=0)
        
        g_starP = sum_ai/sum_bi
        g_starS = g_starP/(1 + sum_ci/sum_di)
        g_starS_values[mask_high] = g_starS
    
    return g_starS_values if len(g_starS_values) > 1 else g_starS_values[0]

# Example usage:
T_values = np.array([1.77/30])  # Temperatures in GeV
print(g_starS(T_values))


14.926972901348597
